In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Preprocessing and Data Augmentation
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_data = datasets.ImageFolder('../data/train', transform=transform)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class BananaNet(nn.Module):
    def __init__(self, num_classes=2):
        super(BananaNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 56 * 56, 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x))) # 112x112
        x = self.pool(F.relu(self.conv2(x))) # 56x56
        x = x.view(-1, 32 * 56 * 56) # Flatten
        x = F.relu(self.fc1(x)) # 32x56x56 -> 128
        x = self.dropout(x)
        x = self.fc2(x) # 128 -> num_classes
        return x

model = BananaNet().to(device)

In [9]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_one_epoch():
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()   # Reset gradients
        outputs = model(inputs) # Forward pass
        loss = criterion(outputs, labels)
        loss.backward()         # Backward pass (autograd)
        optimizer.step()        # Update weights

        running_loss += loss.item()
    return running_loss / len(train_loader)

for epoch in range(10):
    loss = train_one_epoch()
    print(f"Epoch {epoch+1}, Training Loss: {loss:.4f}")

# After training:
torch.save(model.state_dict(), '../saved_models/BananaNet.pth')

Epoch 1, Training Loss: 0.2654
Epoch 2, Training Loss: 1.0631
Epoch 3, Training Loss: 0.5273
Epoch 4, Training Loss: 0.2338
Epoch 5, Training Loss: 0.2472
Epoch 6, Training Loss: 0.0857
Epoch 7, Training Loss: 0.0976
Epoch 8, Training Loss: 0.0428
Epoch 9, Training Loss: 0.1021
Epoch 10, Training Loss: 0.0759
